# RAG Evaluation Pipeline
Αξιολόγηση του RAG συστήματος με RAGAS + Excel

**Βήμα 1:** Φόρτωση ερωτήσεων/απαντήσεων από Excel  
**Βήμα 2:** Εκτέλεση RAG pipeline για κάθε ερώτηση  
**Βήμα 3:** Αξιολόγηση με RAGAS metrics  
**Βήμα 4:** Αποθήκευση αποτελεσμάτων

## Cell 1: Εγκατάσταση βιβλιοθηκών

In [ ]:
!pip install ragas openpyxl pandas datasets --quiet

## Cell 2: Imports & Config

In [ ]:
import sys
sys.path.insert(0, ".")

import json
import pandas as pd
from pathlib import Path
from datasets import Dataset

from rag_retriever import RAGRetriever
from config import VECTOR_DB_PATH

EXCEL_PATH       = Path("data/evaluation_questions.xlsx")
RESULTS_PATH     = Path("data/evaluation_results_new.xlsx")
INCREMENTAL_PATH = Path("data/eval_incremental_new.json")

retriever = RAGRetriever()
print(f"Imports OK! Collection: {retriever.count()} documents")


## Cell 3: Δομή του Excel

Το Excel πρέπει να έχει αυτές τις στήλες:

| Question | Answer |
|----------|-------------|
| What is the max spindle speed? | The maximum spindle speed is 24000 RPM |
| What are the safety requirements? | ... |

In [ ]:
# Φόρτωση ερωτήσεων από Excel
df = pd.read_excel(EXCEL_PATH)

# Έλεγχος στηλών
required_cols = ["Question", "Answer"]
for col in required_cols:
    if col not in df.columns:
        raise ValueError(f"Λείπει η στήλη '{col}' από το Excel!")

print(f"Βρέθηκαν {len(df)} ερωτήσεις")
print("\nΠρώτες 5 ερωτήσεις:")
print(df[["Question", "Answer"]].head(5).to_string())

## Cell 4: Φόρτωση Vector DB και Embedding Model

In [ ]:
import importlib
import rag_retriever
importlib.reload(rag_retriever)
from rag_retriever import RAGRetriever
retriever = RAGRetriever()


In [ ]:
# Το retrieve και generate_answer γίνονται πλέον μέσω RAGRetriever
# Test:
# Επιλογή 1: πραγματική ερώτηση
docs   = retriever.retrieve("What is the maximum spindle speed?", n_results=5)
answer = retriever.generate_answer("What is the maximum spindle speed?", docs)
print(f"Pipeline OK — {len(docs)} docs retrieved")



## Cell 5: Εκτέλεση RAG για κάθε ερώτηση

In [ ]:
# Φόρτωση ήδη επεξεργασμένων — συνέχεια από crash
import os
os.remove("data/eval_incremental_new.json")
print("Deleted — θα ξαναρχίσει από το 0")

if INCREMENTAL_PATH.exists():
    with open(INCREMENTAL_PATH, "r", encoding="utf-8") as f:
        saved = json.load(f)
    questions         = saved["questions"]
    ground_truths     = saved["ground_truths"]
    answers           = saved["answers"]
    contexts_list     = saved["contexts_list"]
    processed_indices = set(saved.get("processed_indices", []))
    print(f"Συνέχεια από {len(questions)} ήδη επεξεργασμένες ερωτήσεις...")
else:
    questions = []; ground_truths = []; answers = []; contexts_list = []
    processed_indices = set()

for idx, row in df.iterrows():
    if idx in processed_indices:
        print(f"[{idx+1}/{len(df)}] παρακάμπτεται (ήδη done)")
        continue

    question     = row["Question"]
    ground_truth = row["Answer"]
    print(f"[{idx+1}/{len(df)}] {question[:60]}...")

    docs     = retriever.retrieve(question, n_results=5)
    contexts = [doc["text"] for doc in docs if doc.get("text")]
    answer   = retriever.generate_answer(question, docs)

    questions.append(question); ground_truths.append(ground_truth)
    answers.append(answer);     contexts_list.append(contexts)
    processed_indices.add(idx)
    print(f"  → {answer[:80]}...")

    INCREMENTAL_PATH.write_text(
        json.dumps({"questions": questions, "ground_truths": ground_truths,
                    "answers": answers, "contexts_list": contexts_list,
                    "processed_indices": list(processed_indices)},
                   ensure_ascii=False, indent=2), encoding="utf-8"
    )

print(f"\nΟλοκληρώθηκε! {len(answers)} απαντήσεις.")


In [ ]:
import requests
resp = requests.post(
    "http://localhost:11434/api/generate",
    json={
        "model": "qwen3:8b",
        "prompt": "What is 2+2?",
        "stream": False,
        "think": True,
        "options": {"num_ctx": 512}
    },
    timeout=300
)
data = resp.json()
print("thinking:", data.get("thinking", "NOT PRESENT"))
print("response:", data.get("response"))


## Cell 6: Αξιολόγηση με RAGAS

In [ ]:
!pip install langchain-huggingface --quiet

In [ ]:
import time, requests

start = time.time()
resp = requests.post(
    "http://localhost:11434/api/generate",
    json={
        "model": "qwen3:8b",
        "system": "You are a helpful assistant.",
        "prompt": "What is 2+2? Answer in one word.",
        "stream": False,
        "think": False,
        "options": {"num_ctx": 512, "temperature": 0.1, "num_gpu": 99}
    },
    timeout=300
)
elapsed = time.time() - start
data = resp.json()
print(f"Time: {elapsed:.1f}s")
print(f"Response: {data.get('response')}")
print(f"Think field present: {'thinking' in data}")


In [ ]:
from ragas import evaluate
from ragas.metrics import faithfulness, answer_relevancy, context_recall, context_precision
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_ollama import OllamaLLM  # FIX: non-deprecated import
from langchain_community.embeddings import HuggingFaceEmbeddings
from ragas import RunConfig

run_config = RunConfig(timeout=120, max_retries=2, max_workers=1)
# Truncate contexts πριν δοθούν στο RAGAS (max 2 chunks, 600 chars το καθένα)
truncated_contexts = [[c[:600] for c in ctx[:2]] for ctx in contexts_list]

# FIX: δημιουργία eval_dataset που έλειπε εντελώς
eval_dataset = Dataset.from_dict({
    "question":     questions,
    "answer":       answers,
    "contexts":     truncated_contexts,
    "ground_truth": ground_truths
})
print(f"Eval dataset: {len(eval_dataset)} ερωτήσεις")

# LLM judge
ollama_llm = OllamaLLM(model="llama3.1:8b", base_url="http://localhost:11434", timeout=300,temperature=0, num_gpu=99,system="/no_think")
ragas_llm  = LangchainLLMWrapper(ollama_llm)

# FIX: ίδιο μοντέλο με vectordb
hf_embeddings   = HuggingFaceEmbeddings(model_name="BAAI/bge-base-en-v1.5")
ragas_embeddings = LangchainEmbeddingsWrapper(hf_embeddings)

results = evaluate(
    dataset=eval_dataset,
    metrics=[faithfulness, answer_relevancy, context_recall, context_precision],
    llm=ragas_llm,
    embeddings=ragas_embeddings,
     run_config=run_config
)

print("\n=== ΑΠΟΤΕΛΕΣΜΑΤΑ RAGAS ===")
print(f"Faithfulness:       {results['faithfulness']:.3f}")
print(f"Answer Relevancy:   {results['answer_relevancy']:.3f}")
print(f"Context Recall:     {results['context_recall']:.3f}")
print(f"Context Precision:  {results['context_precision']:.3f}")


## Cell 7: Αποθήκευση αποτελεσμάτων στο Excel

In [ ]:
# Δημιουργία DataFrame με αποτελέσματα
results_df = results.to_pandas()

# Προσθήκη ερωτήσεων και απαντήσεων
results_df.insert(0, "question",     questions)
results_df.insert(1, "ground_truth", ground_truths)
results_df.insert(2, "rag_answer",   answers)

# Αποθήκευση σε Excel
results_df.to_excel(RESULTS_PATH, index=False)
print(f"Αποτελέσματα αποθηκεύτηκαν: {RESULTS_PATH}")

# Συνοπτική εμφάνιση
print("\n=== ΣΥΝΟΨΗ ===")
print(results_df[["question", "faithfulness", "answer_relevancy", 
                   "context_recall", "context_precision"]].to_string())

In [ ]:
import json
with open("data/eval_incremental_new.json", "r", encoding="utf-8") as f:
    saved = json.load(f)

for i, ctx in enumerate(saved["contexts_list"][:3]):
    print(f"Q{i+1}: {len(ctx)} chunks, total chars: {sum(len(c) for c in ctx)}")
